# Corporate KYC Demo — Part 2: Traversal + Graph Algorithms

Companion to [`demo-story.md`](demo-story.md) and [`answer-key.md`](answer-key.md).
Run [`demo.ipynb`](demo.ipynb) (Part 1: Entity Resolution) first — this
notebook assumes the graph is already resolved (14 Organizations, 6
People, 3 RegisteredAgents, 8 Addresses; see Part 1's Section 7 for the
full before/after).

This part has four beats:

1. **Cypher traversal — trace the real owner.** Who actually controls the
   loan applicant, through ownership layers no single document states.
2. **Cypher traversal — find the loop.** A circular ownership structure,
   an obfuscation red flag.
3. **GDS — Louvain community detection.** Find the shell-company ring as a
   cluster, distinct from the legitimate companies and the clean control
   pair, using structure alone.
4. **GDS — Betweenness centrality.** Find the single person bridging the
   ring into a legitimate company's boardroom.

Beats 1-2 are plain Cypher — no algorithm needed, just graph traversal.
Beats 3-4 need GDS because they're global computations (modularity
optimization; all-pairs shortest paths) that no single pattern match can
express. See `demo-story.md`'s "Why graph queries vs. why graph
algorithms" section for the full reasoning.

**On GDS Sessions:** this Aura instance runs GDS as a separate on-demand
Session, not as procedures embedded in the database (see Part 1). This
notebook only projects and computes (`mutate` + `stream`) and visualizes
the results in the notebook itself -- it doesn't write anything back to
AuraDB, which also sidesteps the write-back hang we ran into in Part 1.

In [ ]:
import os
from datetime import timedelta
from dotenv import load_dotenv
from neo4j import GraphDatabase
from graphdatascience.session import AuraAPICredentials, GdsSessions, DbmsConnectionInfo, SessionMemory
import pandas as pd

load_dotenv()

URI = os.environ["NEO4J_URI"]
USERNAME = os.environ["NEO4J_USERNAME"]
PASSWORD = os.environ["NEO4J_PASSWORD"]
DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD), notifications_min_severity="OFF")
driver.verify_connectivity()
print("Connected to", URI, "via the plain Neo4j driver")

SESSION_NAME = "docintel-aga-analysis"

api_credentials = AuraAPICredentials(
    client_id=os.environ["CLIENT_ID"],
    client_secret=os.environ["CLIENT_SECRET"],
    project_id=os.environ.get("TENANT_ID") or None,
)
sessions = GdsSessions(api_credentials=api_credentials)

db_connection = DbmsConnectionInfo(uri=URI, username=USERNAME, password=PASSWORD)

gds = sessions.get_or_create(
    session_name=SESSION_NAME,
    memory=SessionMemory.m_2GB,
    db_connection=db_connection,
    ttl=timedelta(minutes=30),
    neo4j_driver_config={"notifications_min_severity": "OFF"},
)
gds.verify_connectivity()
print("GDS session ready:", SESSION_NAME)

## 1. Cypher traversal — trace the real owner

Anchor Point Partners LLC is the loan applicant. Its own KYC packet lists
only corporate owners (Cobalt Trading Co, Vantage Point Estates LLC) — no
individual person appears anywhere on its own paperwork. A variable-length
traversal on `OWNS` walks past those immediate corporate owners to find who
*actually* controls it, however many layers deep. This is the regulatory
definition of ultimate beneficial ownership (UBO), and it's a plain Cypher
pattern match — no algorithm required.

In [ ]:
def ubo_paths(tx):
    result = tx.run("""
        MATCH p = (person:Person)-[:OWNS*1..6]->(applicant:Organization {legalName: "Anchor Point Partners LLC"})
        RETURN person.name AS beneficialOwner,
               [n IN nodes(p) | coalesce(n.legalName, n.name)] AS path,
               length(p) AS hops
        ORDER BY hops
    """)
    return pd.DataFrame([dict(r) for r in result])

with driver.session(database=DATABASE) as session:
    ubo = session.execute_read(ubo_paths)

ubo

Dorian Voss reaches the loan applicant through two independent ownership
paths — one two hops deep (via his wholly-owned Vantage Point Estates LLC),
one four hops deep (via Meridian Holdings LLC into the Northgate/Cobalt
chain) — despite never appearing on Anchor Point Partners LLC's own
paperwork at all.

Visualize the ownership paths directly -- `neo4j_viz` can build a graph
straight from a Cypher result (`from_neo4j`), not just from a GDS
projection, so plain traversal results get the same visual treatment as
the algorithm output later in this notebook.

In [ ]:
from neo4j_viz.neo4j import from_neo4j

ubo_graph_result = driver.execute_query(
    """
    MATCH p = (person:Person)-[:OWNS*1..6]->(applicant:Organization {legalName: "Anchor Point Partners LLC"})
    RETURN p
    """,
    database_=DATABASE,
)

VG_ubo = from_neo4j(ubo_graph_result)
for node in VG_ubo.nodes:
    node.caption = node.properties.get("legalName") or node.properties.get("name")
    node.color = "#4C72B0" if "Person" in node.properties.get("labels", []) else "#DD8452"
for rel in VG_ubo.relationships:
    rel.caption = f"OWNS {rel.properties.get('pct', '')}"

VG_ubo.render_widget(initial_zoom=1.0)

## 2. Cypher traversal — find the loop

A short cyclic pattern match: does any organization's ownership chain loop
back to itself? Circular ownership (A owns B owns C owns A) is a textbook
obfuscation red flag — again, just a pattern match, not an algorithm.

In [ ]:
def ownership_cycles(tx):
    result = tx.run("""
        MATCH p = (o:Organization)-[:OWNS*2..6]->(o)
        RETURN DISTINCT [n IN nodes(p) | n.legalName] AS cycle, length(p) AS length
    """)
    return pd.DataFrame([dict(r) for r in result])

with driver.session(database=DATABASE) as session:
    cycles = session.execute_read(ownership_cycles)

cycles

Meridian Holdings LLC → Northgate Ventures LLC → Cobalt Trading Co →
Meridian Holdings LLC: a closed ownership loop among three of the six
ring companies.

Visualize the loop directly: three nodes, three `OWNS` edges, forming a
closed triangle.

In [ ]:
cycle_graph_result = driver.execute_query(
    """
    MATCH p = (o:Organization)-[:OWNS*2..6]->(o)
    RETURN p
    LIMIT 1
    """,
    database_=DATABASE,
)

VG_cycle = from_neo4j(cycle_graph_result)
for node in VG_cycle.nodes:
    node.caption = node.properties.get("legalName") or node.properties.get("name")
for rel in VG_cycle.relationships:
    rel.caption = f"OWNS {rel.properties.get('pct', '')}"

VG_cycle.render()

## 3. GDS — Louvain community detection

Project the ownership and officer graph (`OWNS` + `OFFICER_OF`, emitted in
both directions so community detection treats them as undirected
structural connections) into the GDS session, then run Louvain. Louvain
optimizes graph modularity globally — it finds densely-interconnected
clusters without anyone specifying what a "cluster" should look like, which
is exactly why this needs an algorithm and not a query: nobody wrote a
pattern that says "these six companies form a ring" anywhere in the source
documents.

We expect the six shell-ring companies (Meridian, Northgate, Cobalt, Anchor
Point, Ashford Capital, Vantage Point) to fall into one tight community,
distinct from the legitimate operating companies and the unrelated clean
pair.

In [ ]:
if gds.graph.exists("ownershipGraph"):
    gds.graph.drop("ownershipGraph")

G, project_result = gds.graph.project.cypher(
    "ownershipGraph",
    """
    CALL () {
        MATCH (a)-[:OWNS]->(b)
        RETURN a AS source, b AS target
        UNION
        MATCH (a)-[:OWNS]->(b)
        RETURN b AS source, a AS target
        UNION
        MATCH (a)-[:OFFICER_OF]->(b)
        RETURN a AS source, b AS target
        UNION
        MATCH (a)-[:OFFICER_OF]->(b)
        RETURN b AS source, a AS target
    }
    RETURN gds.graph.project.remote(source, target, {
      sourceNodeLabels: labels(source),
      targetNodeLabels: labels(target),
      relationshipType: 'CONNECTED'
    })
    """,
)
print(f"Projected {G.node_count()} nodes, {G.relationship_count()} edges (both directions)")

In [ ]:
louvain_result = gds.louvain.mutate(G, mutate_property="louvainCommunity")
print(f"Louvain found {louvain_result.community_count} communities "
      f"(modularity {louvain_result.modularity:.3f})")

louvain_stream = gds.graph.node_properties.stream(
    G, ["louvainCommunity"], db_node_properties=["legalName", "name"]
)
louvain_stream["entity"] = louvain_stream["legalName"].fillna(louvain_stream["name"])
louvain_stream = louvain_stream[["entity", "louvainCommunity"]].sort_values(
    ["louvainCommunity", "entity"]
)
louvain_stream

Group by community to see the clusters directly:

In [ ]:
for community_id, members in louvain_stream.groupby("louvainCommunity")["entity"]:
    print(f"Community {community_id}: {', '.join(members)}")

Visualize it: an interactive graph colored by Louvain community. The
six-company ring should render as one visually distinct cluster, separate
from the legitimate companies and the clean control pair.

In [ ]:
from neo4j_viz.gds import from_gds

VG = from_gds(gds, G, node_properties=["louvainCommunity"], db_node_properties=["legalName", "name"])
for node in VG.nodes:
    node.caption = node.properties.get("legalName") or node.properties.get("name")
VG.color_nodes(property="louvainCommunity")
VG.render_widget(initial_zoom=0.8)

## 4. GDS — Betweenness centrality

Betweenness centrality requires computing shortest paths between every
pair of nodes and counting how often each node sits on one — a genuinely
global computation, not a local pattern. We expect Priya Nandakumar to
score highest: she's a director of Northgate Ventures LLC (in the ring)
*and* a director of Cascade Robotics Inc (a legitimate company), making
her the sole bridge between those two otherwise-separate networks.

In [ ]:
betweenness_result = gds.betweenness.mutate(G, mutate_property="betweenness")
print(f"Betweenness computed for {betweenness_result.node_properties_written} nodes")

betweenness_stream = gds.graph.node_properties.stream(
    G, ["betweenness"], db_node_properties=["legalName", "name"]
)
betweenness_stream["entity"] = betweenness_stream["legalName"].fillna(betweenness_stream["name"])
betweenness_stream = betweenness_stream[["entity", "betweenness"]].sort_values(
    "betweenness", ascending=False
)
betweenness_stream.head(10)

Visualize it with `neo4j_viz` again: color by the Louvain community
computed earlier and size by betweenness on the same graph, so both
algorithms' results show up in one view. Priya Nandakumar should be a
small-sized dot (she's not a hub by degree) sitting at a clearly larger
size than her community's other members -- the visual signature of a
bridge: unremarkable locally, structurally important globally.

In [ ]:
VG_bw = from_gds(
    gds, G,
    node_properties=["louvainCommunity", "betweenness"],
    db_node_properties=["legalName", "name"],
)
for node in VG_bw.nodes:
    node.caption = node.properties.get("legalName") or node.properties.get("name")
VG_bw.color_nodes(property="louvainCommunity")
VG_bw.resize_nodes(property="betweenness")
VG_bw.render_widget(initial_zoom=0.8)

## 5. Payoff

Recap what each step surfaced, and that none of it was stated in any
single source document:

- The UBO trace showed Dorian Voss controlling the loan applicant through
  two paths, neither disclosed on the applicant's own paperwork.
- The cycle check showed three ring companies owning each other in a loop.
- Louvain found the full six-company ring as one visually distinct
  community, structurally separate from the legitimate and clean clusters,
  without anyone telling it what a "ring" looks like.
- Betweenness centrality found the one person bridging the ring into a
  legitimate company's boardroom.

This notebook only computes and visualizes -- nothing here writes back to
AuraDB. Clean up the GDS session when you're done (it's billed while it
exists):

In [ ]:
G.drop()
sessions.delete(session_name=SESSION_NAME)
print("Session deleted:", SESSION_NAME)